# Module 3: Deploy — Deep Agents + LangSmith Deployments

> Part of the **Modular Workshops** series. Standalone, ~15 min.

Ship the deep agent in `agents/deep_agent/` to **LangSmith Deployments** with the `deepagents` CLI. The deployed agent gets 30+ endpoints out of the box — REST, MCP, A2A, Agent Protocol, HITL, memory, and Studio UI.

**What you'll do:**
1. Inspect the deployable agent layout (`agents/deep_agent/`)
2. Run it locally with `deepagents dev`
3. Dry-run and then actually `deepagents deploy` to LangSmith


## Setup


In [1]:
import sys
from pathlib import Path
project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from dotenv import load_dotenv
load_dotenv(dotenv_path=project_root / ".env", override=True)

import os
print("LANGSMITH_API_KEY set:", bool(os.environ.get("LANGSMITH_API_KEY")))
print("OPENAI_API_KEY set:   ", bool(os.environ.get("OPENAI_API_KEY")))
print("TAVILY_API_KEY set:   ", bool(os.environ.get("TAVILY_API_KEY")))


LANGSMITH_API_KEY set: True
OPENAI_API_KEY set:    True
TAVILY_API_KEY set:    True


## 1. Project Structure

A deployable deep agent is just a directory with a few files. We already have one at `agents/deep_agent/`.


In [2]:
import os

agent_dir = str(project_root / "agents" / "deep_agent")
for root, dirs, files in os.walk(agent_dir):
    # Skip __pycache__ for clarity
    dirs[:] = [d for d in dirs if d != "__pycache__"]
    level = root.replace(agent_dir, "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    for f in files:
        print(f"{indent}  {f}")


deep_agent/
  deepagents.toml
  agent.py
  AGENTS.md
  skills/
    linkedin-post/
      SKILL.md
    twitter-post/
      SKILL.md


## 2. Configuration

`deepagents.toml` is the deploy config. Minimal: a name and a model.

`AGENTS.md` is the agent's identity — loaded into the prompt via `memory=` and editable by the agent itself.


In [3]:
# deepagents.toml -- the deploy configuration
toml_path = os.path.join(agent_dir, "deepagents.toml")
with open(toml_path) as f:
    print("deepagents.toml:")
    print(f.read())

# AGENTS.md -- the agent's identity
agents_path = os.path.join(agent_dir, "AGENTS.md")
with open(agents_path) as f:
    print("AGENTS.md:")
    print(f.read())


deepagents.toml:
[agent]
name = "modular-workshops-deep-agent"
model = "openai:gpt-4.1-mini"

AGENTS.md:
# Research Assistant

You are an expert research assistant that can search the web, synthesize findings, and produce polished reports and content.

## Workflow

1. **Plan** — Use `write_todos` to break the task into steps
2. **Research** — Delegate research to the `research-agent` using the `task()` tool
3. **Synthesize** — Combine findings into a comprehensive report
4. **Write** — Save the final report to `/final_report.md`
5. **Remember** — Save key takeaways to `/memories/research_notes.md` for future reference

## Rules

- Delegate research to the research-agent rather than searching directly
- After receiving research results, synthesize and write the report yourself
- Consolidate citations — each unique URL gets one number [1], [2], [3]
- End reports with a Sources section listing all referenced URLs
- Check for relevant skills when asked to create specific content formats (e

The agent itself lives in `agent.py` (next to `deepagents.toml`). Open it for the source — note that the `agent` variable at module level is what gets deployed, and `langgraph.json` at the workshop root points to it.


## 3. Local Development

Three CLI commands you'll use:

```bash
# Scaffold a new agent project
deepagents init my-agent

# Run locally for development (Studio UI + hot reload)
deepagents dev --config agents/deep_agent/deepagents.toml --port 2024

# Deploy to LangSmith
deepagents deploy --config agents/deep_agent/deepagents.toml
```

Running `deepagents dev` gives you the Studio UI at <http://localhost:2024> — useful to step through tool calls and approve HITL interrupts visually.


## 4. Dry-Run a Deploy

`--dry-run` builds the deploy artifacts without uploading. Use it to sanity-check what would ship.


In [4]:
import subprocess
result = subprocess.run(
    ["deepagents", "deploy", "--config", "deepagents.toml", "--dry-run"],
    cwd=agent_dir,
    capture_output=True, text=True,
)
print(result.stdout or result.stderr)




  Agent: modular-workshops-deep-agent
  Model: openai:gpt-4.1-mini

  Memory seed (1 file(s)):
    /AGENTS.md

  Skills seed (2 file(s)):
    /linkedin-post/SKILL.md
    /twitter-post/SKILL.md

  Sandbox: none

  Build directory: /var/folders/jj/2fvdkyfj0856p6_6sdvv74rw0000gn/T/deepagents-deploy-yoabo13v
  Generated files: _seed.json, deploy_graph.py, langgraph.json, pyproject.toml

Dry run — artifacts generated but not deployed.
Inspect the build directory: /var/folders/jj/2fvdkyfj0856p6_6sdvv74rw0000gn/T/deepagents-deploy-yoabo13v



## 5. Deploy to LangSmith (Optional)

Run the cell below to deploy the agent to **LangSmith Deployments**. The deployment takes a few minutes and runs in the background — feel free to start Module 4 while it deploys.

> Requires a LangSmith API key with deployment permissions (a `lsv2_sk_...` service key, not a personal token).


In [5]:
# Deploy to LangSmith (requires deployment permissions on your API key)
result = subprocess.run(
    ["deepagents", "deploy", "--config", "deepagents.toml"],
    cwd=agent_dir,
    capture_output=True, text=True,
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)


Note: 'langgraph deploy' is in beta. Expect frequent updates and improvements.

⚠️  Security Recommendation: Consider switching to Wolfi Linux for enhanced security.
   Wolfi is a security-oriented, minimal Linux distribution designed for containers.
   To switch, add '"image_distro": "wolfi"' to your langgraph.json config file.
   Saved deployment name to /private/var/folders/jj/2fvdkyfj0856p6_6sdvv74rw0000gn/T/deepagents-deploy-ldxtfjfr/.env
Docker is installed but not running.
Start Docker and try again.
Using remote build instead.

1. Looking up deployment 'modular-workshops-deep-agent'
   No deployment found. Will create.
2. Creating deployment 'modular-workshops-deep-agent'
   Deployment ID: cdbc08df-7360-4666-8fa0-7fb03867a11d
3. Creating source archive
   Archive created (0.0 MB)
4. Requesting upload URL
5. Uploading source

   Uploading (0.0 MB)... 100%
6. Triggering remote build
   View status: https://smith.langchain.com/o/58636190-0252-4526-9dc7-0b09b37b499c/host/deployment

## 6. What You Get with LangSmith Deployments

Once deployed, your agent is reachable through 30+ endpoints — you build it once, the platform exposes it everywhere:

| Capability | What you can do |
|---|---|
| **REST API** | Standard HTTP requests against `/runs`, `/threads`, `/store` |
| **Studio UI** | Visual debugger to step through state, threads, and tool calls |
| **Agent Protocol** | Stream runs and pause for human input |
| **MCP server** | Other agents can call your agent as a tool |
| **A2A** | Agent-to-agent calls with handoffs |
| **Persistent Store** | `/memories/` survives restarts and threads (via the platform's Store) |
| **HITL** | Interrupt and resume from any client |
| **Cron / Scheduled runs** | Trigger your agent on a schedule |


## Deploy Recap

| File | Purpose |
|------|---------|
| `deepagents.toml` | Agent name, model, sandbox config |
| `agent.py` | The deployable `agent` object (built with `create_deep_agent`) |
| `AGENTS.md` | Agent identity and instructions (loaded via `memory=`) |
| `skills/` | On-demand capability modules (loaded via `skills=`) |
| `langgraph.json` | Project-level graph registration (workshop root) |

**Next:** Module 4 covers how to trace, evaluate, and monitor the deployed agent with LangSmith.
